01_eda.ipynb

## 1. Загрузка данных

In [ ]:
import sys
sys.path.append("..")
from src.data_loader import load_pkl, load_or_build_cache

pkl_data = load_pkl()
label_cache = load_or_build_cache()

In [ ]:
import h5py

## 2. Общая структура корпуса

In [ ]:
print(pkl_data["train"]["id"][:5])

In [ ]:
label_lookup = {} 

with h5py.File("../data/raw/mosei.hdf5", "r") as f:
    grp = f["All Labels"]
    for key in grp.keys():
        video_id, seg_str = key.rsplit("[", 1)
        seg_idx = int(seg_str.rstrip("]"))
        feats = grp[key]["features"][0]  # (1,7) -> (7,)
        label_lookup[(video_id, seg_idx)] = feats

print("Всего записей:", len(label_lookup))
sample_key = next(iter(label_lookup))
print(sample_key, label_lookup[sample_key])

## 3. Сшивка sentiment и эмоций

In [ ]:
for split in ["train", "valid", "test"]:
    n = len(label_cache[split]["mask"])
    matched = label_cache[split]["mask"].sum()
    print(f"{split}: {matched}/{n} ({100*matched/n:.1f}%) сматчено")

Несматченные сэмплы (~5-10%) остаются валидны для sentiment, для emotion-головы маскируются.

## 4. Распределение sentiment

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os

os.makedirs("../figures/eda", exist_ok=True)

CLASS_NAMES = ["negative", "neutral", "positive"]
SPLITS = ["train", "valid", "test"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

raw = label_cache["train"]["sentiment_raw"]
axes[0].hist(raw, bins=np.arange(-3.25, 3.5, 0.25), color="#7F77DD", edgecolor="white")
axes[0].axvline(0, color="black", linestyle="--", linewidth=1)
axes[0].set_title("Распределение sentiment-скора (train)")
axes[0].set_xlabel("sentiment score [-3, 3]")
axes[0].set_ylabel("количество сэмплов")

class_counts = {s: np.bincount(label_cache[s]["sentiment_class"], minlength=3) for s in SPLITS}

x = np.arange(3)
width = 0.25
for i, s in enumerate(SPLITS):
    counts = class_counts[s]
    pct = counts / counts.sum() * 100
    axes[1].bar(x + i * width, pct, width=width, label=s)

axes[1].set_xticks(x + width)
axes[1].set_xticklabels(CLASS_NAMES)
axes[1].set_ylabel("% сэмплов")
axes[1].set_title("Доля классов sentiment по сплитам")
axes[1].legend()

plt.tight_layout()
plt.savefig("../figures/eda/sentiment_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

print("Train sentiment class distribution:")
for name, cnt in zip(CLASS_NAMES, class_counts["train"]):
    print(f"  {name:10s}: {cnt:6d} ({100*cnt/class_counts['train'].sum():.1f}%)")

Есть ощутимый дисбаланс класса positive - будем использовать метрику F1

## 5. Частоты co-occurence эмоций

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

EMOTION_NAMES = ["happy", "sad", "anger", "surprise", "disgust", "fear"]

mask = label_cache["train"]["mask"]
emo = label_cache["train"]["emotion_binary"][mask]

freq_pct = emo.mean(axis=0) * 100

plt.figure(figsize=(6, 4))
plt.bar(EMOTION_NAMES, freq_pct, color="#1D9E75")
plt.ylabel("% сэмплов с присутствием эмоции")
plt.title(f"Частота эмоций (train, N={emo.shape[0]})")
plt.tight_layout()
plt.savefig("../figures/eda/emotion_frequency.png", dpi=150, bbox_inches="tight")
plt.show()

for name, pct in zip(EMOTION_NAMES, freq_pct):
    print(f"  {name:10s}: {pct:5.1f}%")

co_matrix = emo.T @ emo

plt.figure(figsize=(6, 5))
plt.imshow(co_matrix, cmap="viridis")
plt.colorbar(label="число совместных случаев")
plt.xticks(range(6), EMOTION_NAMES, rotation=45)
plt.yticks(range(6), EMOTION_NAMES)
plt.title("Co-occurrence эмоций (train)")
for i in range(6):
    for j in range(6):
        val = int(co_matrix[i, j])
        color = "white" if val < co_matrix.max() / 2 else "black"
        plt.text(j, i, val, ha="center", va="center", color=color, fontsize=9)
plt.tight_layout()
plt.savefig("../figures/eda/emotion_cooccurrence.png", dpi=150, bbox_inches="tight")
plt.show()

n_emotions = emo.sum(axis=1)  # 0..6 эмоций на сэмпл
plt.figure(figsize=(5, 3.5))
plt.hist(n_emotions, bins=np.arange(-0.5, 7.5, 1), color="#534AB7", edgecolor="white")
plt.xlabel("число эмоций на сэмпл")
plt.ylabel("количество сэмплов")
plt.title("Кардинальность эмоциональных меток")
plt.tight_layout()
plt.savefig("../figures/eda/emotion_cardinality.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nСэмплов без единой эмоции (все нули): {(n_emotions == 0).sum()} ({100*(n_emotions==0).mean():.1f}%)")
print(f"Среднее число эмоций на сэмпл: {n_emotions.mean():.2f}")

Доминирование happy - прямое следствие того же позитивного перекоса, что мы видели в sentiment. 

В co-occurrence интересны связи между негативными эмоциями: anger + disgust - 1576 случаев, это 55.6% от всех disgust-сэмплов и 46.6% от anger. sad↔disgust тоже сильная пара - 47.5% от disgust. Это совпадает с психологической интуицией: anger, disgust и sadness образуют кластер "негативного affect", который часто проявляется вместе в одном высказывании.

fear - самая редкая и самая "изолированная" эмоция (максимум пересечений — с happy/sad, по 600, и это всё ещё немного относительно её базовой частоты 1294). Для multilabel-классификатора это будет самый сложный класс — мало примеров, слабые паттерны совместного появления, на которые модель могла бы опереться.

## 6. Корреляция сентиментов и эмоций

In [ ]:
mask = label_cache["train"]["mask"]
emo = label_cache["train"]["emotion_binary"][mask]
sent_class = label_cache["train"]["sentiment_class"][mask]
sent_raw = label_cache["train"]["sentiment_raw"][mask]

EMOTION_NAMES = ["happy", "sad", "anger", "surprise", "disgust", "fear"]
CLASS_NAMES = ["negative", "neutral", "positive"]

# --- A. Доля эмоции внутри каждого класса sentiment ---
presence_by_class = np.zeros((6, 3))
for c in range(3):
    sub = emo[sent_class == c]
    presence_by_class[:, c] = sub.mean(axis=0) * 100

plt.figure(figsize=(6, 5))
plt.imshow(presence_by_class, cmap="magma", aspect="auto")
plt.colorbar(label="% присутствия эмоции внутри класса")
plt.xticks(range(3), CLASS_NAMES)
plt.yticks(range(6), EMOTION_NAMES)
plt.title("Эмоции по классам sentiment (train)")
for i in range(6):
    for j in range(3):
        val = presence_by_class[i, j]
        color = "white" if val < presence_by_class.max() / 2 else "black"
        plt.text(j, i, f"{val:.0f}%", ha="center", va="center", color=color, fontsize=9)
plt.tight_layout()
plt.savefig("../figures/eda/sentiment_emotion_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

# --- B. Корреляция непрерывного sentiment с каждой бинарной эмоцией ---
correlations = np.array([np.corrcoef(sent_raw, emo[:, i])[0, 1] for i in range(6)])

plt.figure(figsize=(6, 4))
colors = ["#1D9E75" if c > 0 else "#D85A30" for c in correlations]
plt.barh(EMOTION_NAMES, correlations, color=colors)
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("корреляция с sentiment score")
plt.title("Связь эмоций с непрерывным sentiment (train)")
plt.tight_layout()
plt.savefig("../figures/eda/sentiment_emotion_correlation.png", dpi=150, bbox_inches="tight")
plt.show()

for name, corr in zip(EMOTION_NAMES, correlations):
    print(f"  {name:10s}: r = {corr:+.3f}")

Знаки полностью совпадают с ожиданием по валентности, и это даёт нам последнюю, самую сильную проверку качества сшивки эмоций: если бы где-то в assignment-матчинге проскочила перепутанная пара, мы скорее всего увидели бы хотя бы одну "перевёрнутую" корреляцию - её нет, все знаки логичны.

## 7. Длины последовательностей и паддинг 

In [ ]:
text = pkl_data["train"]["text"]
audio = pkl_data["train"]["audio"]
vision = pkl_data["train"]["vision"]

combined_zero = (
    np.all(text == 0, axis=-1) &
    np.all(audio == 0, axis=-1) &
    np.all(vision == 0, axis=-1)
) 

real_length = (~combined_zero).sum(axis=1) 

print("Реальная длина — min/max/mean/median:",
      real_length.min(), real_length.max(),
      round(real_length.mean(), 2), np.median(real_length))

for i in [0, 1, 2]:
    mask_row = combined_zero[i].astype(int)
    print(f"Сэмпл {i} (реальная длина={real_length[i]}): {mask_row}")

In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(real_length, bins=30, color="#378ADD", edgecolor="white")
plt.xlabel("реальная длина последовательности")
plt.ylabel("количество сэмплов")
plt.title("Распределение длины высказываний (train)")
plt.tight_layout()
plt.savefig("../figures/eda/sequence_length_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

padding_ratio = 1 - real_length / 50
print(f"\nСредняя доля паддинга: {padding_ratio.mean()*100:.1f}%")
print(f"Сэмплов короче 10 шагов: {(real_length < 10).sum()} ({100*(real_length<10).mean():.1f}%)")
print(f"Сэмплов, заполняющих всю длину 50: {(real_length == 50).sum()} ({100*(real_length==50).mean():.1f}%)")

In [ ]:
real_mask = ~combined_zero

audio_valid = audio[real_mask] 
vision_valid = vision[real_mask]

print("Audio (COVAREP)  — min/max/mean/std:",
      round(audio_valid.min(), 2), round(audio_valid.max(), 2),
      round(audio_valid.mean(), 3), round(audio_valid.std(), 3))
print("Vision (FACET)   — min/max/mean/std:",
      round(vision_valid.min(), 2), round(vision_valid.max(), 2),
      round(vision_valid.mean(), 3), round(vision_valid.std(), 3))

In [ ]:
print("Audio: isnan =", np.isnan(audio_valid).sum(), "| isinf =", np.isinf(audio_valid).sum())
print("Vision: isnan =", np.isnan(vision_valid).sum(), "| isinf =", np.isinf(vision_valid).sum())

if np.isinf(audio_valid).sum() > 0:
    inf_mask = np.isinf(audio_valid)
    cols = np.where(inf_mask.any(axis=0))[0]
    print("Колонки audio с inf:", cols)
    for c in cols:
        n = inf_mask[:, c].sum()
        print(f"  колонка {c}: {n} inf-значений ({100*n/audio_valid.shape[0]:.2f}%)")

In [ ]:
def finite_stats(arr, name):
    finite = np.isfinite(arr)
    bad = (~finite).sum()
    vals = arr[finite]
    print(f"{name}: non-finite={bad} ({100*bad/arr.size:.3f}%), "
          f"min={vals.min():.2f} max={vals.max():.2f} mean={vals.mean():.3f} std={vals.std():.3f}")

finite_stats(audio_valid, "Audio (COVAREP)")
finite_stats(vision_valid, "Vision (FACET)")

In [ ]:
from scipy import stats as sps

def describe(arr, name):
    print(f"{name}: n={len(arr)}, mean={np.mean(arr):.3f}, median={np.median(arr):.3f}, "
          f"std={np.std(arr):.3f}, min={np.min(arr):.3f}, max={np.max(arr):.3f}")

sent_raw_train = label_cache["train"]["sentiment_raw"]
describe(sent_raw_train, "sentiment_raw")
mode_sent = sps.mode(np.round(sent_raw_train, 2), keepdims=False)
print(f"  мода (округлено до 0.01): {mode_sent.mode:+.2f} ({mode_sent.count} раз)")

describe(real_length, "real_length")
mode_len = sps.mode(real_length, keepdims=False)
print(f"  мода длины: {mode_len.mode} шагов ({mode_len.count} раз)")

In [ ]:
from scipy import stats as sps

audio_clean = audio_valid.copy()
finite_col7 = audio_clean[np.isfinite(audio_clean[:, 7]), 7]
audio_clean[~np.isfinite(audio_clean[:, 7]), 7] = finite_col7.min()  # тот же fix, что решили для inf

audio_z = sps.zscore(audio_clean, axis=0)

plt.figure(figsize=(14, 4))
plt.boxplot([audio_z[:, i] for i in range(audio_z.shape[1])], showfliers=False)
plt.xlabel("индекс признака COVAREP")
plt.ylabel("z-score")
plt.title("Разброс признаков COVAREP после стандартизации")
plt.xticks(rotation=90, fontsize=6)
plt.axhline(0, color="gray", linewidth=0.5)
plt.tight_layout()
plt.savefig("../figures/eda/audio_feature_scale_boxplot_standardized.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
import seaborn as sns

audio_corr = np.corrcoef(audio_clean, rowvar=False)

plt.figure(figsize=(7, 6))
plt.imshow(audio_corr, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar(label="корреляция Пирсона")
plt.title("Корреляция между признаками COVAREP (train)")
plt.tight_layout()
plt.savefig("../figures/eda/audio_feature_correlation.png", dpi=150, bbox_inches="tight")
plt.show()

triu_idx = np.triu_indices(74, k=1)
corrs = audio_corr[triu_idx]
top_idx = np.argsort(-np.abs(corrs))[:10]
print("Топ-10 самых скоррелированных пар аудио-признаков:")
for idx in top_idx:
    i, j = triu_idx[0][idx], triu_idx[1][idx]
    print(f"  признак {i} – признак {j}: r = {corrs[idx]:+.3f}")

emotion_names = ['happy', 'sad', 'anger', 'surprise', 'disgust', 'fear']

plt.figure(figsize=(6, 5))
sns.heatmap(
    emo_corr,
    annot=True,          
    fmt='.2f',
    cmap='coolwarm',
    vmin=-1, vmax=1,
    square=True,
    linewidths=0.5,
    cbar_kws={'label': 'Pearson correlation'},
    xticklabels=emotion_names,
    yticklabels=emotion_names
)

plt.title('Correlation between emotions (train, binary labels)', fontsize=12)
plt.tight_layout()
plt.savefig('../figures/eda/emotion_correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## Выводы

Корпус CMU-MOSEI содержит 16265/1869/4643 сэмплов в train/valid/test, с тремя синхронизированными модальностями (текст — GloVe 300d, аудио — COVAREP 74d, видео — FACET 35d), выровненными по словам до фиксированной длины 50 шагов.

Метка sentiment присутствует напрямую в исходном файле признаков, тогда как шесть эмоциональных меток (happy, sad, anger, surprise, disgust, fear) хранятся в отдельном источнике без общего позиционного ключа с основным файлом. Прямое сопоставление по порядковому индексу сегмента дало лишь 45.5% совпадений; переформулировка задачи как задачи о назначениях (минимизация расхождения sentiment между источниками, алгоритм Хангариана) повысила покрытие до 94.8% на train (99.0% на valid, 97.0% на test) при нулевом числе принятых пар с расхождением sentiment выше порога 0.05. Несматченные сэмплы (~5–10%) сохраняют валидность для задачи sentiment и помечаются маской для исключения из лосса emotion-головы.
Распределение sentiment демонстрирует умеренный дисбаланс с перекосом в сторону positive (49.3% против 29.0% negative и 21.7% neutral), что обосновывает выбор macro-F1, а не accuracy, как основной метрики — модель, предсказывающая константный класс positive, достигла бы accuracy около 49% без единого верного предсказания на оставшихся классах.

Среди эмоций happy доминирует (53.1% присутствия), что согласуется с общим позитивным смещением корпуса; fear (8.4%) и surprise (10.0%) — наиболее редкие классы. Анализ co-occurrence выявил выраженный кластер совместного появления негативных эмоций (anger, disgust, sad), тогда как happy слабо пересекается с этой группой — что подтверждает осмысленность multilabel-постановки задачи. Среднее число одновременных эмоций на сэмпл — 1.37, что в одном порядке величины с цифрами, приводимыми в литературе по этому корпусу. Корреляция каждой эмоции с непрерывным sentiment-скором подтвердила ожидаемые знаки по валентности (happy: r=+0.531; disgust: r=−0.564; sad: r=−0.345; anger: r=−0.385; surprise и fear — близки к нулю), что одновременно служит независимой проверкой корректности процедуры сопоставления меток.

Анализ длины последовательностей показал среднюю долю паддинга 50.3% при фиксированной длине 50, с 7.2% сэмплов, заполняющих её полностью, и 5.5% — короче 10 шагов. Обнаружена локализованная проблема качества данных: 1249 значений (0.31% одной из 74 колонок COVAREP, вероятно логарифм частоты основного тона) являются бесконечными на безголосых кадрах — это требует замены на конечное значение перед нормализацией признаков на этапе построения baseline-модели.

In [ ]:
import h5py

with h5py.File("../data/raw/mosei.hdf5", "r") as f:
    print("Группы верхнего уровня в файле:")
    for key in f.keys():
        print(" -", key)

In [ ]:
from src.features import load_words_index, build_transcripts, extract_text_tfidf_features

words_flat = load_words_index("../data/raw/mosei.hdf5")

transcripts_by_split = {}
for split in ["train", "valid", "test"]:
    ids = label_cache[split]["ids"]
    matched_idx = label_cache[split]["hdf5_idx"]
    real_len = real_length if split == "train" else None  # real_length у нас сейчас только для train, см. ниже
    transcripts, word_counts = build_transcripts(ids, matched_idx, words_flat, real_length=real_len)
    transcripts_by_split[split] = transcripts

tfidf_features, vectorizer = extract_text_tfidf_features(transcripts_by_split)
print("\nРазмер словаря TF-IDF:", len(vectorizer.vocabulary_))
print("Пример транскрипта:", transcripts_by_split["train"][0])

In [ ]:
import h5py
from collections import defaultdict\

from src.features import (
    load_words_index, build_transcripts, extract_text_tfidf_features,
    get_real_mask,
)

text = pkl_data["train"]["text"]
audio_t = pkl_data["train"]["audio"]
vision_t = pkl_data["train"]["vision"]
mask_t = get_real_mask(text, audio_t, vision_t)
real_len_train = mask_t.sum(axis=1)

ids = label_cache["train"]["ids"]
matched_idx = label_cache["train"]["hdf5_idx"]
valid = matched_idx != -1

raw_counts = np.zeros(len(ids), dtype=np.int32)
filtered_counts = np.zeros(len(ids), dtype=np.int32)

for i in np.where(valid)[0]:
    vid = ids[i, 0]
    words = words_flat.get((vid, matched_idx[i]))
    if words is None:
        continue
    raw_counts[i] = len(words)
    filtered_counts[i] = len([w for w in words if w.strip() and w.lower() != "sp"])

print("=== Тест 1: raw (со 'sp') vs filtered (без 'sp') ===")
print("raw      — корреляция:", round(np.corrcoef(raw_counts[valid], real_len_train[valid])[0,1], 3),
      "| медиана |diff|:", np.median(np.abs(raw_counts[valid] - real_len_train[valid])))
print("filtered — корреляция:", round(np.corrcoef(filtered_counts[valid], real_len_train[valid])[0,1], 3),
      "| медиана |diff|:", np.median(np.abs(filtered_counts[valid] - real_len_train[valid])))

# --- Тест 2: совпадает ли число сегментов на видео между 'words' и 'All Labels' ---
with h5py.File("../data/raw/mosei.hdf5", "r") as f:
    words_by_video = defaultdict(int)
    for key in f["words"].keys():
        vid, _ = key.rsplit("[", 1)
        words_by_video[vid] += 1
    labels_by_video = defaultdict(int)
    for key in f["All Labels"].keys():
        vid, _ = key.rsplit("[", 1)
        labels_by_video[vid] += 1

common = set(words_by_video) & set(labels_by_video)
mismatched = sum(1 for v in common if words_by_video[v] != labels_by_video[v])
print(f"\n=== Тест 2: согласованность сегментации между группами ===")
print(f"Видео в обоих группах: {len(common)}")
print(f"С разным числом сегментов: {mismatched} ({100*mismatched/len(common):.1f}%)")

In [ ]:
from src.features import build_feature_cache

feature_cache = build_feature_cache(pkl_data, label_cache, "../data/raw/mosei.hdf5", force_rebuild=True)

for name, splits in feature_cache.items():
    if isinstance(splits, dict) and "train" in splits and isinstance(splits["train"], dict):
        # audio: вложенный словарь {mfcc, prosody}
        for sub_name, arr in splits["train"].items():
            print(f"{name}.{sub_name}: train shape = {arr.shape}")
    else:
        print(f"{name}: train shape = {splits['train'].shape}")